# Análisis de resultados Gold

Este notebook consulta las tablas Gold creadas por `src/gold.ipynb`. No crea ni modifica tablas. Las consultas describen asociaciones observadas entre Twitch y Steam; no permiten afirmar causalidad.

In [ ]:
dbutils.widgets.text("catalog", "steam_game_intelligence")
dbutils.widgets.text("gold_schema", "gold")
catalog = dbutils.widgets.get("catalog")
gold_schema = dbutils.widgets.get("gold_schema")

## 1. Cobertura de las tablas Gold

Muestra el volumen y las ventanas temporales disponibles para saber qué datos sustentan las consultas posteriores.

In [ ]:
%sql
SELECT 'game_growth' AS table_name, COUNT(*) AS row_count,
       COUNT(DISTINCT app_id) AS distinct_games,
       MIN(month_start) AS first_month,
       MAX(month_start) AS last_month
FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.game_growth')
UNION ALL
SELECT 'twitch_steam_relationship', COUNT(*), COUNT(DISTINCT app_id),
       MIN(month_start), MAX(month_start)
FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.twitch_steam_relationship')
UNION ALL
SELECT 'dim_game', COUNT(*), COUNT(DISTINCT app_id), NULL, NULL
FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.dim_game');

## 2. Juegos con mayor crecimiento mensual

Resume los juegos y meses con mayor crecimiento consecutivo. Se excluyen valores nulos y se limita la muestra para facilitar la revisión.

In [ ]:
%sql
SELECT app_id, name, month_start, avg_players,
       prev_avg_players, mom_change_pct, months_since_release
FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.game_growth')
WHERE mom_change_pct IS NOT NULL
ORDER BY mom_change_pct DESC, app_id, month_start
LIMIT 50;

## 3. Evolución mensual agregada

Calcula el promedio y la mediana de jugadores por mes. Sirve para observar la evolución general sin confundirla con el comportamiento de un solo juego.

In [ ]:
%sql
SELECT month_start,
       COUNT(DISTINCT app_id) AS games_with_measurement,
       AVG(avg_players) AS mean_avg_players,
       percentile_approx(avg_players, 0.5) AS median_avg_players,
       AVG(mom_change_pct) AS mean_mom_change_pct
FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.game_growth')
GROUP BY month_start
ORDER BY month_start;

## 4. Asociación entre Twitch y el crecimiento posterior en Steam

Agrupa las observaciones con jugadores disponibles en `t` y `t+1`. La correlación y los promedios son descriptivos: no controlan por lanzamientos, promociones, estacionalidad ni otros factores.

In [ ]:
%sql
SELECT
  COUNT(*) AS observations,
  COUNT(DISTINCT app_id) AS distinct_games,
  AVG(hours_watched) AS mean_hours_watched,
  AVG(growth_pct_t_to_t1) AS mean_steam_growth_t_to_t1,
  percentile_approx(growth_pct_t_to_t1, 0.5) AS median_steam_growth_t_to_t1,
  corr(hours_watched, growth_pct_t_to_t1) AS corr_hours_growth
FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.twitch_steam_relationship')
WHERE avg_players_t IS NOT NULL
  AND avg_players_t1 IS NOT NULL
  AND growth_pct_t_to_t1 IS NOT NULL;

## 5. Asociación por nivel de exposición en Twitch

Divide las observaciones en cuartiles de horas vistas y compara el crecimiento posterior de Steam entre grupos. El resultado sigue siendo asociativo, no causal.

In [ ]:
%sql
WITH observations AS (
  SELECT *, ntile(4) OVER (ORDER BY hours_watched, app_id, month_start) AS exposure_quartile
  FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.twitch_steam_relationship')
  WHERE avg_players_t IS NOT NULL
    AND avg_players_t1 IS NOT NULL
    AND growth_pct_t_to_t1 IS NOT NULL
)
SELECT exposure_quartile,
       COUNT(*) AS observations,
       MIN(hours_watched) AS min_hours_watched,
       MAX(hours_watched) AS max_hours_watched,
       AVG(growth_pct_t_to_t1) AS mean_growth_t_to_t1,
       percentile_approx(growth_pct_t_to_t1, 0.5) AS median_growth_t_to_t1
FROM observations
GROUP BY exposure_quartile
ORDER BY exposure_quartile;

## 6. Catálogo y proporción de reseñas

Consulta la dimensión Gold para revisar la distribución de precios, propietarios y valoración. Los valores nulos se conservan como ausencia de información.

In [ ]:
%sql
SELECT is_unreleased,
       COUNT(*) AS games,
       AVG(price) AS mean_price,
       AVG(review_ratio) AS mean_review_ratio,
       COUNT_IF(review_ratio IS NULL) AS games_without_review_ratio
FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.dim_game')
GROUP BY is_unreleased
ORDER BY is_unreleased;